In [ ]:
!pip install gdown -q
!gdown --folder "https://drive.google.com/drive/folders/1oE8A3YCEcDeTiv9Yc2QtzEUBX2XyLDtW"

Retrieving folder contents
Processing file 1cp--e02t83bcRu9Umuvv6973l7IUexlg features.parquet
Processing file 1tZ7OoGvs7hm_soSYBYNZYhUk3g4QzlFt returns.parquet
Processing file 1fcOeGk0OL6xw7tWq9MsdKbaut47u9eTa universe.parquet
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1cp--e02t83bcRu9Umuvv6973l7IUexlg
From (redirected): https://drive.google.com/uc?id=1cp--e02t83bcRu9Umuvv6973l7IUexlg&confirm=t&uuid=7afd00c8-6052-4f9a-9bea-3c227eb89793
To: /content/Data Sets/features.parquet
100% 1.69G/1.69G [00:29<00:00, 57.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tZ7OoGvs7hm_soSYBYNZYhUk3g4QzlFt
To: /content/Data Sets/returns.parquet
100% 55.6M/55.6M [00:00<00:00, 59.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1fcOeGk0OL6xw7tWq9MsdKbaut47u9eTa
To: /content/Data Sets/universe.parquet
100% 1.19M/1.19M [00:00<00:00, 105MB/s]
Download completed


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
features = pd.read_parquet("/content/Data Sets/features.parquet")
returns = pd.read_parquet("/content/Data Sets/returns.parquet")
universe = pd.read_parquet("/content/Data Sets/universe.parquet")

print("Features shape:", features.shape)
print("Returns shape:", returns.shape)
print("Universe shape:", universe.shape)

Features shape: (5058, 47674)
Returns shape: (3775, 2167)
Universe shape: (5058, 2167)


In [ ]:
print("Features columns sample:")
print(features.columns[:10].tolist())
print("\nFeatures index sample:")
print(features.index[:5].tolist())

Features columns sample:
[('macd', 1), ('macd', 2), ('macd', 3), ('macd', 4), ('macd', 5), ('macd', 6), ('macd', 7), ('macd', 8), ('macd', 9), ('macd', 10)]

Features index sample:
[Timestamp('2005-01-03 00:00:00'), Timestamp('2005-01-04 00:00:00'), Timestamp('2005-01-05 00:00:00'), Timestamp('2005-01-06 00:00:00'), Timestamp('2005-01-07 00:00:00')]


In [ ]:
print("Returns columns sample:", returns.columns[:5].tolist())
print("Returns index sample:", returns.index[:3].tolist())
print("\nUniverse sample (first 5 stocks, first 3 days):")
print(universe.iloc[:3, :5])

Returns columns sample: [1, 2, 3, 4, 5]
Returns index sample: [Timestamp('2005-01-03 00:00:00'), Timestamp('2005-01-04 00:00:00'), Timestamp('2005-01-05 00:00:00')]

Universe sample (first 5 stocks, first 3 days):
            1  2  3  4  5
Date                     
2005-01-03  0  0  0  0  0
2005-01-04  0  0  0  0  0
2005-01-05  0  0  0  0  0


In [ ]:
avg_tradeable = universe.sum(axis=1).mean()
print(f"Average tradeable stocks per day: {avg_tradeable:.0f}")

print("\nTradeable stocks on last date:")
last_date = universe.index[-1]
print(f"Date: {last_date}, Count: {universe.loc[last_date].sum()}")

Average tradeable stocks per day: 996

Tradeable stocks on last date:
Date: 2025-02-07 00:00:00, Count: 1000


In [ ]:
trend = features['trend_1_3']
print(trend.shape)
print(trend.iloc[:3, :5])

(5058, 2167)
                   1         2   3         4         5
Date                                                  
2005-01-03       NaN       NaN NaN       NaN       NaN
2005-01-04 -1.414214 -1.414214 NaN -1.414214 -1.414214
2005-01-05  0.785016  0.525942 NaN  0.364820  0.936002


In [ ]:
date = pd.Timestamp('2020-01-02')
signal = trend.loc[date]
print(signal.head())

1    1.136118
2   -0.531697
3    0.598328
4    1.336008
5   -0.906529
Name: 2020-01-02 00:00:00, dtype: float64


In [ ]:
tradeable = universe.loc[date]
signal_filtered = signal[tradeable == 1]
print(signal_filtered.shape)

(1000,)


In [ ]:
signal_filtered = signal_filtered.dropna()
print(signal_filtered.shape)

(1000,)


In [ ]:
ranked = signal_filtered.rank(pct=True)
print(ranked.head())
print(ranked.min(), ranked.max())

1    0.687
2    0.331
3    0.534
5    0.261
6    0.840
Name: 2020-01-02 00:00:00, dtype: float64
0.001 1.0


In [ ]:
n_stocks = 50
top_stocks = ranked.nlargest(n_stocks)
bottom_stocks = ranked.nsmallest(n_stocks)

print("Top stocks:", top_stocks.head())
print("Bottom stocks:", bottom_stocks.head())

Top stocks: 1638    1.000
49      0.999
2023    0.998
48      0.997
1053    0.996
Name: 2020-01-02 00:00:00, dtype: float64
Bottom stocks: 954     0.001
1889    0.002
1857    0.003
2010    0.004
1396    0.005
Name: 2020-01-02 00:00:00, dtype: float64


In [ ]:
weights = pd.Series(0.0, index=signal_filtered.index)
weights[top_stocks.index] = 1 / n_stocks
weights[bottom_stocks.index] = -1 / n_stocks

print("Sum of weights:", weights.sum())
print("Number of long positions:", (weights > 0).sum())
print("Number of short positions:", (weights < 0).sum())

Sum of weights: -2.7755575615628914e-17
Number of long positions: 50
Number of short positions: 50


In [ ]:
signal_demeaned = signal_filtered - signal_filtered.mean()
print(signal_demeaned.sum())

5.861977570020827e-14


In [ ]:
weights_scored = signal_demeaned / signal_demeaned.abs().sum() * 2

print("Sum of weights:", weights_scored.sum())
print("Long exposure:", weights_scored[weights_scored>0].sum())
print("Short exposure:", weights_scored[weights_scored<0].sum())

Sum of weights: 1.249000902703301e-16
Long exposure: 1.0
Short exposure: -1.0


In [ ]:
def get_weights(history, today_universe):
    if len(history) == 0:
        return {}

    latest_date = history.index[-1]
    signal = history['trend_1_3'].loc[latest_date]

    tradeable = today_universe[today_universe == 1].index
    signal = signal[signal.index.isin(tradeable)].dropna()

    if len(signal) == 0:
        return {}

    signal_demeaned = signal - signal.mean()

    if signal_demeaned.abs().sum() == 0:
        return {}

    weights = signal_demeaned / signal_demeaned.abs().sum()  # book value = 1
    weights = weights.clip(-0.1, 0.1)  # max weight constraint

    return weights.to_dict()

In [ ]:
!pip install tqdm -q

from tqdm import tqdm

dates = universe.index
stock_ids = universe.columns
weights_df = pd.DataFrame(0.0, index=dates, columns=stock_ids)

for t in tqdm(dates):
    history = features.loc[features.index < t]
    today_universe = universe.loc[t]
    raw_weights = get_weights(history, today_universe)
    for stock_id, w in raw_weights.items():
        if stock_id in stock_ids and today_universe.get(stock_id, 0) == 1:
            weights_df.loc[t, stock_id] = w

print("Backtest done!")
print("Weights shape:", weights_df.shape)

100%|██████████| 5058/5058 [49:36<00:00,  1.70it/s]

Backtest done!
Weights shape: (5058, 2167)


In [ ]:
# Compute PnL
common_idx = weights_df.index.intersection(returns.index)
common_cols = weights_df.columns.intersection(returns.columns)

w = weights_df.loc[common_idx, common_cols]
r = returns.loc[common_idx, common_cols]

gross_pnl = (w * r).sum(axis=1)

# Trading costs
traded = weights_df.diff().abs().sum(axis=1).fillna(weights_df.iloc[0].abs().sum())
net_pnl = gross_pnl - 0.0001 * traded

# Sharpe ratios
gross_sharpe = np.sqrt(252) * gross_pnl.mean() / gross_pnl.std()
net_sharpe = np.sqrt(252) * net_pnl.mean() / net_pnl.std()

print(f"Gross Sharpe: {gross_sharpe:.4f}")
print(f"Net Sharpe: {net_sharpe:.4f}")

Gross Sharpe: -1.2593
Net Sharpe: -1.5231


In [ ]:
test_dates = universe.index[2000:2200]  # dates around 2012-2013
weights_test2 = pd.DataFrame(0.0, index=test_dates, columns=stock_ids)

for t in tqdm(test_dates):
    history = features.loc[features.index < t]
    today_universe = universe.loc[t]
    raw_weights = get_weights(history, today_universe)
    for stock_id, w in raw_weights.items():
        if stock_id in stock_ids and today_universe.get(stock_id, 0) == 1:
            weights_test2.loc[t, stock_id] = w

print("Done!")

100%|██████████| 200/200 [01:48<00:00,  1.84it/s]

Done!


In [ ]:
common_idx = weights_test2.index.intersection(returns.index)
common_cols = weights_test2.columns.intersection(returns.columns)

w = weights_test2.loc[common_idx, common_cols]
r = returns.loc[common_idx, common_cols]

gross_pnl_test2 = (w * r).sum(axis=1)
traded_test2 = weights_test2.diff().abs().sum(axis=1).fillna(weights_test2.iloc[0].abs().sum())
net_pnl_test2 = gross_pnl_test2 - 0.0001 * traded_test2

gross_sharpe_test2 = np.sqrt(252) * gross_pnl_test2.mean() / gross_pnl_test2.std()
net_sharpe_test2 = np.sqrt(252) * net_pnl_test2.mean() / net_pnl_test2.std()

print(f"Gross Sharpe: {gross_sharpe_test2:.4f}")
print(f"Net Sharpe: {net_sharpe_test2:.4f}")

Gross Sharpe: 0.0192
Net Sharpe: -0.4223


In [ ]:
print("Weights test dates:", weights_test.index[0], "→", weights_test.index[-1])
print("Returns dates:", returns.index[0], "→", returns.index[-1])

NameError: name 'weights_test' is not defined

In [ ]:
weights_df2 = pd.DataFrame(0.0, index=dates, columns=stock_ids)

for t in tqdm(dates):
    history = features.loc[features.index < t]
    today_universe = universe.loc[t]
    raw_weights = get_weights(history, today_universe)
    for stock_id, w in raw_weights.items():
        if stock_id in stock_ids and today_universe.get(stock_id, 0) == 1:
            weights_df2.loc[t, stock_id] = w

print("Done!")